# Insightia — Notebook 03 (REWRITE) : Bloc 3 — Analyse contextuelle

**Objectif** : comprendre **dans quelles conditions** les problèmes clients apparaissent.

✅ On ne cherche pas la cause technique.
✅ On identifie des **hotspots** (où ça se concentre) : canal, device, contexte, région, support, urgence.

**Question** : *« Dans quels cas ces problèmes surviennent-ils ? »*

## Sorties
- `outputs/comments_clean.parquet` (mis à jour si besoin, avec `situations`)
- `outputs/block3_cross_<dimension>.csv`
- `outputs/block3_hotspots.csv`
- `outputs/block3_summary.json`


## 1) Chargement des données (issues du Bloc 2)

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PARQUET = OUT_DIR / "comments_clean.parquet"
if not DATA_PARQUET.exists():
    raise FileNotFoundError("outputs/comments_clean.parquet introuvable. Exécute d'abord Notebook 02.")

df = pd.read_parquet(DATA_PARQUET)

print("Nb lignes:", len(df))
display(df.head(3))


## 2) Contrôles colonnes (et FIX automatique de `situations`)

In [ ]:

import re
import unicodedata

REQUIRED = {"motif","sentiment","commentaire","date"}
missing = sorted(list(REQUIRED - set(df.columns)))
print("Colonnes requises manquantes:", missing)

# --- utilitaires texte (reconstruction situations si besoin) ---
def strip_accents(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", s) if unicodedata.category(ch) != "Mn")

def normalize_text(s: str) -> str:
    s = str(s).lower().strip()
    s = strip_accents(s)
    s = s.replace("’","'").replace("'","")
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

STOPWORDS_LIGHT = {
    "le","la","les","un","une","des","de","du","au","aux","et","ou","mais","donc","or",
    "je","tu","il","elle","on","nous","vous","ils","elles","dans","sur","pour","par","avec","sans",
    "ce","cet","cette","ces","qui","que","quoi","dont"
}
STOPWORDS_STRUCT = {
    "cest","pas","jai","suis","etre","avoir","tout","rien","tres","trop","encore","toujours",
    "juste","peux","peut","fait","faire","vais","aller","merci","bonjour","soir","jour","fois",
    "quand","meme","alors","est"
}

def tokenize_clean(s: str):
    toks = normalize_text(s).split()
    return [
        t for t in toks
        if len(t) >= 3 and (t not in STOPWORDS_LIGHT) and (t not in STOPWORDS_STRUCT) and (not t.isdigit())
    ]

# Règles simples mais robustes (tu peux les affiner)
SITUATIONS_RULES = {
    "perte_temps": {"perds","temps","minutes","heure","journee","fou"},
    "essais_multiples": {"essais","plusieurs","tentatives","reessayer","malgre","encore"},
    "blocage_parcours": {"bloque","blocage","tourne","rond","passe","navance","ecran","formulaire","valider","etape"},
    "recours_support": {"support","appel","telephone","chat","email","joindre","aide","reponse"},
}

def detect_situations(text: str):
    toks = set(tokenize_clean(text))
    hits = []
    for name, words in SITUATIONS_RULES.items():
        # règle: au moins 2 mots du set pour éviter la sur-détection
        if len(toks.intersection(words)) >= 2:
            hits.append(name)
    return hits

if "situations" not in df.columns:
    df["situations"] = df["commentaire"].astype(str).apply(detect_situations)
    print("✅ Colonne `situations` reconstruite (elle manquait).")
else:
    # sécurité type : on force list si jamais c'est une string
    if df["situations"].dtype == object:
        def _ensure_list(x):
            if isinstance(x, list): return x
            if pd.isna(x): return []
            # si c'est un texte "a,b,c"
            if isinstance(x, str):
                x = x.strip()
                if x == "": return []
                return [t.strip() for t in x.split(",") if t.strip()]
            return []
        df["situations"] = df["situations"].apply(_ensure_list)
    print("✅ Colonne `situations` déjà présente (et normalisée).")

# Sauvegarde parquet enrichie pour les notebooks suivants
df.to_parquet(DATA_PARQUET, index=False)
print("✅ Parquet mis à jour:", DATA_PARQUET)

# aperçu
display(df[["motif","sentiment","situations","commentaire"]].head(5))


## 3) Préparation dates & dimensions contextuelles

In [ ]:

# Date → datetime + mois (utile pour Bloc 4 aussi)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).copy()
df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()

CONTEXT_COLS = ["canal", "device", "contexte", "region", "urgence", "support"]
usable_context = [c for c in CONTEXT_COLS if c in df.columns]

print("Dimensions contextuelles utilisables:", usable_context)


## 4) Mise au format long : 1 ligne = (commentaire × situation)

In [ ]:

rows = []

for _, r in df.iterrows():
    sits = r["situations"] if isinstance(r["situations"], list) else []
    for s in sits:
        rec = {
            "motif": r["motif"],
            "situation": s,
            "sentiment": r["sentiment"],
            "month": r["month"],
        }
        for c in usable_context:
            rec[c] = r.get(c, "UNKNOWN")
        rows.append(rec)

df_long = pd.DataFrame(rows)

print("Nb lignes format long:", len(df_long))
display(df_long.head(10))


## 5) Croisements : motif × situation × dimension

In [ ]:

def cross_tab(df_in, cols, min_n=30):
    return (
        df_in.groupby(cols)
             .size()
             .reset_index(name="n")
             .query("n >= @min_n")
             .sort_values("n", ascending=False)
    )

cross_results = {}
for dim in usable_context:
    ct = cross_tab(df_long, ["motif","situation", dim], min_n=30)
    cross_results[dim] = ct
    ct.to_csv(OUT_DIR / f"block3_cross_{dim}.csv", index=False)
    print(f"{dim}: {len(ct)} lignes exportées → outputs/block3_cross_{dim}.csv")

# aperçu
if usable_context:
    display(cross_results[usable_context[0]].head(15))


## 6) Hotspots (liste unique triée par volume)

In [ ]:

hotspots = []
for dim, ct in cross_results.items():
    for _, r in ct.iterrows():
        hotspots.append({
            "motif": r["motif"],
            "situation": r["situation"],
            "dimension": dim,
            "value": r[dim],
            "n": int(r["n"]),
        })

hotspots = pd.DataFrame(hotspots).sort_values("n", ascending=False)
display(hotspots.head(25))

hotspots.to_csv(OUT_DIR / "block3_hotspots.csv", index=False)
print("✅ Export:", OUT_DIR / "block3_hotspots.csv")


## 7) Résumé & traçabilité

In [ ]:

summary = {
    "n_rows": int(len(df)),
    "n_rows_long": int(len(df_long)),
    "period_min": str(df["date"].min().date()),
    "period_max": str(df["date"].max().date()),
    "usable_context": usable_context,
    "outputs": {
        **{f"cross_{d}": f"outputs/block3_cross_{d}.csv" for d in usable_context},
        "hotspots": "outputs/block3_hotspots.csv",
        "data_enriched": "outputs/comments_clean.parquet",
    }
}

with open(OUT_DIR / "block3_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

OUT_DIR / "block3_summary.json"


✅ Fin Notebook 03 (rewrite).

**Next** : Notebook 04 (tendances) utilisera `month` + `situations`.
